# 🎢 Polynomial & Regularized Regression

Linear regression assumes the relationship between features and the target is a perfectly straight line. But what if it's a curve? What if we have too many features and our model starts "memorizing" the data (overfitting)?

Enter **Polynomial Regression** and **Regularization (Ridge & Lasso)**.

## 1. Polynomial Regression (Theory & Math)

If your data curves like a smile (U-shape), a straight line will be a terrible fit. We can trick Linear Regression into fitting a curve by adding polynomial features (squaring or cubing our inputs).

$$ y = \beta_0 + \beta_1x + \beta_2x^2 $$

Even though $x^2$ is non-linear, the equation is still *linear with respect to the weights ($\beta$)*. So, standard Linear Regression can solve it!

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error


In [ ]:
# 1. Generate Non-Linear Synthetic Data
np.random.seed(42)
X = np.sort(np.random.rand(100, 1) * 10, axis=0)
# True function: A sine wave with some noise
y = np.sin(X).ravel() + np.random.normal(0, 0.3, 100)

plt.scatter(X, y, color='blue', alpha=0.5, label='Data')
plt.title('Non-Linear Data')
plt.show()

In [ ]:
# 2. Fit a straight line (Underfitting)
lin_reg = LinearRegression()
lin_reg.fit(X, y)

# 3. Fit a Polynomial Curve (Degree 3)
# make_pipeline automatically creates x^2 and x^3 features, then runs Linear Regression
poly_reg = make_pipeline(PolynomialFeatures(degree=3), LinearRegression())
poly_reg.fit(X, y)

plt.scatter(X, y, color='blue', alpha=0.5)
plt.plot(X, lin_reg.predict(X), color='red', linestyle='--', label='Straight Line (Degree 1)')
plt.plot(X, poly_reg.predict(X), color='green', linewidth=2, label='Polynomial (Degree 3)')
plt.legend()
plt.show()

##  2. The Danger of Overfitting

If we increase the polynomial degree to something crazy like 15, the model will connect almost every dot. It will have a near-perfect score on the training data, but it will fail miserably on new data. This is **Overfitting**.

In [ ]:
# Fit a wildly complex curve (Degree 15)
overfit_reg = make_pipeline(PolynomialFeatures(degree=15), LinearRegression())
overfit_reg.fit(X, y)

X_plot = np.linspace(0, 10, 500).reshape(-1, 1)
plt.scatter(X, y, color='blue', alpha=0.5)
plt.plot(X_plot, overfit_reg.predict(X_plot), color='red', label='Overfitted Curve (Degree 15)')
plt.ylim(-2, 2)
plt.legend()
plt.show()
print('Notice how the red line goes crazy at the edges just to hit specific points!')

## 🛡️ 3. Regularization (Ridge & Lasso)

How do we stop a model from overfitting? We **penalize** it for being too complex. We force the coefficients ($\beta$) to be as small as possible.

1. **Ridge Regression (L2 Penalty)**: Adds the *squared sum* of coefficients to the cost function. It shrinks all coefficients down, but rarely to exactly zero.
2. **Lasso Regression (L1 Penalty)**: Adds the *absolute sum* of coefficients to the cost function. It can shrink useless coefficients to **exactly zero**, acting as automatic Feature Selection!

In [ ]:
# Applying Ridge and Lasso to our overfitted Degree 15 model

# Note: Always scale data before using Regularization!
ridge_reg = make_pipeline(PolynomialFeatures(degree=15), StandardScaler(), Ridge(alpha=1.0))
ridge_reg.fit(X, y)

lasso_reg = make_pipeline(PolynomialFeatures(degree=15), StandardScaler(), Lasso(alpha=0.1))
lasso_reg.fit(X, y)

plt.figure(figsize=(10, 6))
plt.scatter(X, y, color='blue', alpha=0.3)
plt.plot(X_plot, overfit_reg.predict(X_plot), color='red', linestyle=':', label='No Regularization (Crazy)')
plt.plot(X_plot, ridge_reg.predict(X_plot), color='green', label='Ridge (Smoothed)')
plt.plot(X_plot, lasso_reg.predict(X_plot), color='purple', linestyle='--', label='Lasso (Smoothed & Sparse)')
plt.ylim(-2, 2)
plt.legend()
plt.show()

##  Summary
- Use **Polynomial Regression** when the relationship is visibly curved.
- Use **Regularization (Ridge/Lasso)** to tame complex models and prevent overfitting.
- **Lasso** is great if you have 1,000 features and want the algorithm to automatically drop the useless ones by setting their coefficients to 0.